# Generating Dataset

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import time
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Model Params

block_size = 20 # what is the maximum context length for predictions? Sequence length
max_iters = 5000
eval_interval = 100
learning_rate = 3e-4
eval_iters = 200
batch_size = 8 # how many independent sequences will we process in parallel?
n_embd = 128 # embedding dimension
n_head = 8 # number of heads
n_layer = 4
dropout = 0.2


# n_embd = 4 # embedding dimension
# n_head = 2 # number of heads
# n_layer = 1
# dropout = 0.2

In [ ]:
words = []

# 1 to 19
ones = [
    "one","two","three","four","five",
    "six","seven","eight","nine","ten",
    "eleven","twelve","thirteen","fourteen","fifteen",
    "sixteen","seventeen","eighteen","nineteen"
]

# tens words (20, 30, ..., 90)
tens = ["twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]

# add 1–19
words.extend(ones)


for t in tens:
    words.append(t)  # exact 20, 30, ...
    for o in ["one","two","three","four","five","six","seven","eight","nine"]:
        words.append(t)
        words.append(o)


for h in ["one","two","three","four","five","six","seven","eight","nine"]:
    words.append(h)
    words.append("hundred")
    # 101–119, etc.
    for o in ones :
      words.append(h)
      words.append("hundred")
      words.append(o)


    for t in tens:
        words.append(h)
        words.append("hundred")
        words.append(t)
        for o in ["one","two","three","four","five","six","seven","eight","nine"]:
            words.append(h)
            words.append("hundred")
            words.append(t)
            words.append(o)

print(words[1000:1100])   # preview first 200 tokens
print(len(words))    # total vocabulary size


['hundred', 'twenty', 'seven', 'three', 'hundred', 'twenty', 'eight', 'three', 'hundred', 'twenty', 'nine', 'three', 'hundred', 'thirty', 'three', 'hundred', 'thirty', 'one', 'three', 'hundred', 'thirty', 'two', 'three', 'hundred', 'thirty', 'three', 'three', 'hundred', 'thirty', 'four', 'three', 'hundred', 'thirty', 'five', 'three', 'hundred', 'thirty', 'six', 'three', 'hundred', 'thirty', 'seven', 'three', 'hundred', 'thirty', 'eight', 'three', 'hundred', 'thirty', 'nine', 'three', 'hundred', 'forty', 'three', 'hundred', 'forty', 'one', 'three', 'hundred', 'forty', 'two', 'three', 'hundred', 'forty', 'three', 'three', 'hundred', 'forty', 'four', 'three', 'hundred', 'forty', 'five', 'three', 'hundred', 'forty', 'six', 'three', 'hundred', 'forty', 'seven', 'three', 'hundred', 'forty', 'eight', 'three', 'hundred', 'forty', 'nine', 'three', 'hundred', 'fifty', 'three', 'hundred', 'fifty', 'one', 'three', 'hundred', 'fifty', 'two']
3510


In [ ]:
set(words)

{'eight',
 'eighteen',
 'eighty',
 'eleven',
 'fifteen',
 'fifty',
 'five',
 'forty',
 'four',
 'fourteen',
 'hundred',
 'nine',
 'nineteen',
 'ninety',
 'one',
 'seven',
 'seventeen',
 'seventy',
 'six',
 'sixteen',
 'sixty',
 'ten',
 'thirteen',
 'thirty',
 'three',
 'twelve',
 'twenty',
 'two'}

In [ ]:
# Vocabulary:
vocab = set(words)

# Make word -> index and index -> word mappings
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for i, w in enumerate(vocab)}

vocab_size = len(vocab)
print(vocab_size)

28


In [ ]:
for _ in range(10):
  words.extend(words)

In [ ]:
len(words)

3594240

In [ ]:
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ' '.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

In [ ]:
# encode(["one","hundred"])
decode([0,1])

'thirteen nineteen'

In [ ]:
data = torch.tensor(encode(words), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]


In [ ]:
len(val_data)

359424

In [ ]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
x,y = get_batch('train')

In [ ]:
x,y

(tensor([[17, 16, 12, 26, 24, 12, 26, 24,  3, 12, 26, 24, 22, 12, 26, 24, 25, 12,
          26, 24],
         [16, 22, 26, 24, 22, 26, 24,  3, 22, 26, 24, 22, 22, 26, 24, 25, 22, 26,
          24,  8],
         [26,  2, 25, 26, 15, 25, 26,  1, 25, 26, 18, 25, 26, 18,  3, 25, 26, 18,
          22, 25],
         [18, 16, 12, 26, 17, 12, 26, 17,  3, 12, 26, 17, 22, 12, 26, 17, 25, 12,
          26, 17],
         [17, 12, 25, 26, 17, 27, 25, 26, 17, 13, 25, 26, 17, 16, 25, 26, 24, 25,
          26, 24],
         [26, 18, 16, 26, 18,  3, 16, 26, 18, 22, 16, 26, 18, 25, 16, 26, 18,  8,
          16, 26],
         [14, 26,  9, 14, 26,  9,  3, 14, 26,  9, 22, 14, 26,  9, 25, 14, 26,  9,
           8, 14],
         [14, 26,  5, 14, 26,  0, 14, 26,  7, 14, 26, 19, 14, 26, 20, 14, 26,  2,
          14, 26]], device='cuda:0'),
 tensor([[16, 12, 26, 24, 12, 26, 24,  3, 12, 26, 24, 22, 12, 26, 24, 25, 12, 26,
          24,  8],
         [22, 26, 24, 22, 26, 24,  3, 22, 26, 24, 22, 22, 26, 24, 25, 22

In [ ]:
print("Inputs : ")
for each in x:
  print("Input  : "+decode(each.tolist()))
print("-----")
print("Targets : ")
for each in y:
  print("Target : "+decode(each.tolist()))


Inputs : 
Input  : thirty nine six hundred forty six hundred forty one six hundred forty two six hundred forty three six hundred forty
Input  : nine two hundred forty two hundred forty one two hundred forty two two hundred forty three two hundred forty four
Input  : hundred seventeen three hundred eighteen three hundred nineteen three hundred twenty three hundred twenty one three hundred twenty two three
Input  : twenty nine six hundred thirty six hundred thirty one six hundred thirty two six hundred thirty three six hundred thirty
Input  : thirty six three hundred thirty seven three hundred thirty eight three hundred thirty nine three hundred forty three hundred forty
Input  : hundred twenty nine hundred twenty one nine hundred twenty two nine hundred twenty three nine hundred twenty four nine hundred
Input  : five hundred seventy five hundred seventy one five hundred seventy two five hundred seventy three five hundred seventy four five
Input  : five hundred twelve five hundred thirte

In [ ]:
n = 2
for i in range(len(x[n].tolist())):
  context = decode(x[n].tolist()[0:i+1])
  target = itos[y[n].tolist()[i]]
  print(f"When input is {context}, --> target is {target}")

When input is hundred, --> target is seventeen
When input is hundred seventeen, --> target is three
When input is hundred seventeen three, --> target is hundred
When input is hundred seventeen three hundred, --> target is eighteen
When input is hundred seventeen three hundred eighteen, --> target is three
When input is hundred seventeen three hundred eighteen three, --> target is hundred
When input is hundred seventeen three hundred eighteen three hundred, --> target is nineteen
When input is hundred seventeen three hundred eighteen three hundred nineteen, --> target is three
When input is hundred seventeen three hundred eighteen three hundred nineteen three, --> target is hundred
When input is hundred seventeen three hundred eighteen three hundred nineteen three hundred, --> target is twenty
When input is hundred seventeen three hundred eighteen three hundred nineteen three hundred twenty, --> target is three
When input is hundred seventeen three hundred eighteen three hundred ninetee

#Building and Training

In [ ]:

# -----------------------------------------------------------------------------

def init_params():
    params = {}

    # token + positional embeddings
    params['token_embedding_table'] = nn.Embedding(vocab_size, n_embd).to(device)
    params['position_embedding_table'] = nn.Embedding(block_size, n_embd).to(device)

    # transformer blocks
    for layer in range(n_layer):
        # attention weights
        for h in range(n_head):
            params[f'layer{layer}.head{h}.key']   = nn.Linear(n_embd, n_embd//n_head, bias=False).to(device)
            params[f'layer{layer}.head{h}.query'] = nn.Linear(n_embd, n_embd//n_head, bias=False).to(device)
            params[f'layer{layer}.head{h}.value'] = nn.Linear(n_embd, n_embd//n_head, bias=False).to(device)

        # params[f'layer{layer}.proj'] = nn.Linear(n_embd, n_embd).to(device)
        params[f'layer{layer}.ln1'] = nn.LayerNorm(n_embd).to(device)
        params[f'layer{layer}.ln2'] = nn.LayerNorm(n_embd).to(device)

        # feedforward
        params[f'layer{layer}.ff1'] = nn.Linear(n_embd, 4*n_embd).to(device)
        params[f'layer{layer}.ff2'] = nn.Linear(4*n_embd, n_embd).to(device)

    # final norm + lm head
    params['ln_f'] = nn.LayerNorm(n_embd).to(device)
    params['lm_head'] = nn.Linear(n_embd, vocab_size).to(device)

    return params

In [ ]:
params = init_params() #When training

In [ ]:
tril = torch.tril(torch.ones(block_size, block_size, device=device))

In [ ]:
# -----------------------------------------------------------------------------

def attention(x, layer, params):
    B,T,C = x.shape
    head_outputs = []
    for h in range(n_head):
        k = params[f'layer{layer}.head{h}.key'](x)
        q = params[f'layer{layer}.head{h}.query'](x)
        v = params[f'layer{layer}.head{h}.value'](x)

        wei = q @ k.transpose(-2,-1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(tril[:T,:T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = F.dropout(wei, dropout, training=True)

        out = wei @ v
        head_outputs.append(out)

    out = torch.cat(head_outputs, dim=-1)
    # out = params[f'layer{layer}.proj'](out)
    out = F.dropout(out, dropout, training=True)
    return out

In [ ]:
def feedforward(x, layer, params):
    x = params[f'layer{layer}.ff1'](x)
    x = F.relu(x)
    x = params[f'layer{layer}.ff2'](x)
    x = F.dropout(x, dropout, training=True)
    return x

In [ ]:
def transformer_block(x, layer, params):
    x = x + attention(params[f'layer{layer}.ln1'](x), layer, params)
    x = x + feedforward(params[f'layer{layer}.ln2'](x), layer, params)
    return x

In [ ]:
def gpt_forward(idx, targets=None, params=params):
    B,T = idx.shape

    tok_emb = params['token_embedding_table'](idx)       # (B,T,C)
    pos_emb = params['position_embedding_table'](torch.arange(T, device=device)) # (T,C)
    x = tok_emb + pos_emb

    for layer in range(n_layer):
        x = transformer_block(x, layer, params)

    x = params['ln_f'](x)
    logits = params['lm_head'](x)

    loss = None
    if targets is not None:
        B,T,C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)

    return logits, loss

In [ ]:
@torch.no_grad()
def generate(idx, max_new_tokens):
    input_token_length = idx.shape[1]
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = gpt_forward(idx_cond, targets = None, params = params)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx[:, input_token_length:]

In [ ]:
@torch.no_grad()
def estimate_loss(params):
    out = {}
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = gpt_forward(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean()
    return out

In [ ]:
# collect parameters for optimizer
all_params = [p for p in params.values() if isinstance(p, nn.Module)]
optim_params = []
for m in all_params:
    optim_params += list(m.parameters())
optimizer = torch.optim.AdamW(optim_params, lr=learning_rate)

# Now you can run your training loop using gpt_forward + optimizer

In [ ]:
total_params = sum(p.numel() for p in optim_params)
print(f"Total parameters: {total_params}")

Total parameters: 735516


In [ ]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(params)   # pass datasets explicitly
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch(train_data)           # pass dataset explicitly

    # evaluate the loss
    logits, loss = gpt_forward(xb, yb, params)               # functional forward

    # backprop and optimization
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 3.4565, val loss 3.4581
step 100: train loss 2.0696, val loss 2.0625
step 200: train loss 1.4936, val loss 1.4770
step 300: train loss 1.1596, val loss 1.1551
step 400: train loss 0.9898, val loss 0.9774
step 500: train loss 0.8285, val loss 0.8128
step 600: train loss 0.7048, val loss 0.6854
step 700: train loss 0.6081, val loss 0.6050
step 800: train loss 0.5351, val loss 0.5457
step 900: train loss 0.5062, val loss 0.5020
step 1000: train loss 0.4599, val loss 0.4491
step 1100: train loss 0.4389, val loss 0.4380
step 1200: train loss 0.4206, val loss 0.4159
step 1300: train loss 0.3990, val loss 0.3998
step 1400: train loss 0.3823, val loss 0.3859
step 1500: train loss 0.3655, val loss 0.3729
step 1600: train loss 0.3628, val loss 0.3620
step 1700: train loss 0.3609, val loss 0.3585
step 1800: train loss 0.3543, val loss 0.3568
step 1900: train loss 0.3409, val loss 0.3467
step 2000: train loss 0.3393, val loss 0.3334
step 2100: train loss 0.3304, val loss 0.3320


In [ ]:
def generate_text(prompt_input, max_new_tokens=100):
    # split input into words and take last block_size tokens
    prompt_input_list = prompt_input.split()
    prompt_input_list = prompt_input_list[-block_size:]

    # encode to indices
    prompt_input_stoi_ids = encode(prompt_input_list)
    prompt_final = torch.tensor([prompt_input_stoi_ids], dtype=torch.long, device=device)

    # generate new tokens
    output_ids = generate(prompt_final, max_new_tokens=max_new_tokens)  # call functional generate

    # decode indices back to words
    return decode(output_ids[0].tolist())


In [ ]:
prompt_input = "one two three four five six seven eight nine ten"
print(f'"System Prompt : {prompt_input}')
print('------------')
print(f'"Response :')
generate_text(prompt_input)


"System Prompt : one two three four five six seven eight nine ten
------------
"Response :


'eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen twenty twenty one hundred twenty two twenty three twenty four twenty five twenty one twenty four twenty five twenty six twenty seven twenty eight twenty nine thirty thirty one thirty two thirty three thirty four thirty five thirty six thirty seven thirty seven thirty eight thirty nine forty forty one forty two forty three forty four forty five forty six forty seven forty eight forty nine fifty fifty one fifty two fifty three fifty four fifty five fifty six fifty seven fifty eight fifty nine sixty ten sixty sixty one sixty'